# PDCP · N10 — E4: rendering and model-size ablations + N08 reruns
**Notebook 10 of 12.** Kaggle: **GPU T4 x2** · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 7–8.5 h (one session; resumable).

**Inputs:** `kbs-n3-output` (pairs, drawings, images) · `kbs-n5-output` (E1 answers, P-Def) · `kbs-n6-part2` (MEL counterfactual images + E2 answers).

**E4a — does the picture matter?** The same 200 base pairs as N08 (identical selection), redrawn from the stored coordinates in 7 rendering variants plus the default image:
resolution 384 / 1024 (same look, scaled), small / large nodes, thin / thick edges, coloured nodes with grey edges. Models: Qwen2.5-VL-7B, InternVL-4B, Qwen3-VL-2B; P-Def; both orders.
Measures per variant: accuracy, position bias, lure gap, and **instability** = mean |P(A) variant − P(A) default| per pair, compared with the re-run noise floor (default arm here vs the original E1 answers).

**E4b — model size and quantisation.** New models: Qwen2.5-VL-3B, InternVL3.5-2B, InternVL3.5-8B (4-bit), Qwen2.5-VL-7B in 8-bit (vs 4-bit in E1). Each on the 200 base pairs + their MEL counterfactuals (δ −0.05, +0.10; 8-bit model: base pairs only).
Together with E1/E2 models → accuracy and **compactness bias** vs parameter count, within three model families.

**Reruns from N08 (fixed):** P-FewShot3 (InternVL-4B, Qwen3-VL-2B; examples at 384 px, one query at a time), and for InternVL-4B P-NoReason (96 tokens) and P-ReasonFirst (300 tokens).

**Outputs** (`/kaggle/working/n10_output/`): `results/e4/<model>.jsonl`, `e4_queries_<model>.csv`, `e4_render_summary.csv`, `e4_size_summary.csv`, `e4_rerun_summary.csv`, `e4_status.json`, figures `fig_e4_*.png`, `pdcp_infer.py` v0.5, logs, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N03_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"
N05_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n5-output"
N06_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2"
RESUME_ROOT = None
OUTPUT_DIR = "/kaggle/working/n10_output"
N_BASE = 200
SEED = 2026
VARIANTS = {   # rendering variants (default image = N03 PNG, 768 px, node_r 6, edge_w 1.5)
    "res384":     dict(res=384, node_r=3.0, edge_w=0.75),
    "res1024":    dict(res=1024, node_r=8.0, edge_w=2.0, supersample=2),
    "node_small": dict(res=768, node_r=3.0),
    "node_large": dict(res=768, node_r=10.0),
    "edge_thin":  dict(res=768, edge_w=0.75),
    "edge_thick": dict(res=768, edge_w=3.0),
    "colour":     dict(res=768, node_fill="#1f77b4", edge_col="#777777"),
}
RENDER_MODELS = ["Q25VL-7B", "InternVL-4B", "Q3VL-2B"]
SIZE_MODELS = ["InternVL-8B", "Q25VL-3B", "InternVL-2B", "Q25VL-7B-int8"]     # 8-bit model: base pairs only
RERUNS = {"InternVL-4B": ["P-FewShot3", "P-NoReason", "P-ReasonFirst"], "Q3VL-2B": ["P-FewShot3"]}
MEL_DELTAS = [-0.05, 0.10]
PARAMS_B = {"Q3VL-2B": 2, "Q3VL-4B": 4, "Q25VL-3B": 3, "Q25VL-7B": 7, "Q25VL-7B-int8": 7, "InternVL-2B": 2, "InternVL-4B": 4, "InternVL-8B": 8}
FAMILY = {"Q3VL-2B": "Qwen3-VL", "Q3VL-4B": "Qwen3-VL", "Q25VL-3B": "Qwen2.5-VL", "Q25VL-7B": "Qwen2.5-VL", "Q25VL-7B-int8": "Qwen2.5-VL",
          "InternVL-2B": "InternVL3.5", "InternVL-4B": "InternVL3.5", "InternVL-8B": "InternVL3.5"}
SESSION_HOURS = 11.3
N_CPU = 4
DRY_RUN = False      # True = no GPU: write random answers (tests the analysis only)
FAST = False

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib
T0 = time.time()
for d in ["results/e4", "logs", "images", "fewshot_img"]: os.makedirs(f"{OUTPUT_DIR}/{d}", exist_ok=True)
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from PIL import Image
if not DRY_RUN:
    import torch
    N_GPU = torch.cuda.device_count(); assert N_GPU >= 1, "Set Accelerator = GPU T4 x2"
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.57", "accelerate", "bitsandbytes", "qwen-vl-utils", "num2words"], capture_output=True, text=True)
else:
    N_GPU = 2
def find(root, name):
    f = sorted(glob.glob(f"{root}/**/{name}", recursive=True)); assert f, f"{name} not found under {root}"; return os.path.dirname(f[0])
N03 = find(N03_ROOT, "pairs.parquet"); N06 = find(N06_ROOT, "mel_cf_pairs.parquet")
shutil.copy(f"{N06}/pdcp_utils.py", f"{OUTPUT_DIR}/pdcp_utils.py"); sys.path.insert(0, OUTPUT_DIR); import pdcp_utils as U; print("utils", U.__version__)
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet"); MCP = pd.read_parquet(f"{N06}/mel_cf_pairs.parquet")
png = dict(zip(DR.drawing_id, DR.png))
def path(did): return f"{N03}/{png[did]}" if did in png else f"{N06}/images/768/{did}.png"
print("N03:", N03, "| N06:", N06)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

utils 0.1.3
N03: /kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output/n03_output | N06: /kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2/n06_output


## 1. Base pairs (same 200 as N08), MEL counterfactuals, few-shot examples

In [3]:
SUB = PR[(PR.set == "main") & PR.subset400].copy()
BASE = SUB.groupby(["dstress_design", "cond"], group_keys=False).sample(frac=min(1.0, N_BASE / len(SUB)), random_state=SEED)
if len(BASE) < N_BASE: BASE = pd.concat([BASE, SUB.drop(BASE.index).sample(min(N_BASE - len(BASE), len(SUB) - len(BASE)), random_state=SEED)])
BASE = BASE.head(N_BASE)
if FAST: BASE = BASE.head(12)
MEL = MCP[MCP.base_pair.isin(BASE.pair_id) & MCP.cf_delta.isin(MEL_DELTAS) & (MCP.cf_proxy == "mel")]
print(len(BASE), "base pairs |", len(MEL), "MEL counterfactual pairs")
FS = PR[PR.set == "fewshot"]; Di = DR.set_index("drawing_id")
FA = FS[FS.correct == "A"].sort_values("dstress_design", ascending=False)
pick = [FA.iloc[0], FA.iloc[min(1, len(FA) - 1)], (FS[FS.correct == "same"].iloc[0] if (FS.correct == "same").any() else FA.iloc[-1])]
fewshot = []
def small(p):
    q = f"{OUTPUT_DIR}/fewshot_img/{os.path.basename(p)}"; Image.open(p).convert("RGB").resize((384, 384), Image.LANCZOS).save(q) if not DRY_RUN or os.path.exists(p) else None; return q
for k, r in enumerate(pick):
    a, b = path(r.drawing_A), path(r.drawing_B); ka, kb = Di.loc[r.drawing_A, "ksm"], Di.loc[r.drawing_B, "ksm"]
    if k == 1: a, b, ka, kb = b, a, kb, ka
    ans = "same" if r.correct == "same" else ("1" if ka < kb else "2")
    reason = (f"Both drawings have the same stress ({ka:.2f})." if ans == "same" else
              f"Image {ans} has lower stress ({min(ka, kb):.2f} vs {max(ka, kb):.2f}): its drawn distances match path distances more consistently.")
    fewshot.append({"img1": small(a), "img2": small(b), "answer_json": json.dumps({"answer": ans, "confidence": 5, "reason": reason})})
json.dump(fewshot, open(f"{OUTPUT_DIR}/fewshot.json", "w"), indent=1)

200 base pairs | 384 MEL counterfactual pairs


## 2. Render the 7 variants from the stored coordinates

In [4]:
def render_variant(did, v):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, pdcp_utils as U
    out = f"{OUTPUT_DIR}/images/{v}/{did}.png"
    if os.path.exists(out): return out
    r = Di.loc[did]; gc = U.GraphCache(U.make_graph(r.family, int(r.n), int(r.graph_seed)))
    U.render_png(gc, np.array(r.coords).reshape(-1, 2), out, **VARIANTS[v]); return out
for v in VARIANTS: os.makedirs(f"{OUTPUT_DIR}/images/{v}", exist_ok=True)
dids = sorted(set(BASE.drawing_A) | set(BASE.drawing_B))
Parallel(n_jobs=N_CPU, verbose=1)(delayed(render_variant)(d, v) for v in VARIANTS for d in dids)
print(len(dids) * len(VARIANTS), "variant images |", f"{(time.time()-T0)/60:.1f} min")
# mean edge length of every drawing used (N03 drawings do not store it)
def mel_of(did):
    r = Di.loc[did]; gc = U.GraphCache(U.make_graph(r.family, int(r.n), int(r.graph_seed))); return U.mean_edge_length(np.array(r.coords).reshape(-1, 2), gc.E)
MELV = {d: mel_of(d) for d in dids}

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    4.1s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:   13.2s
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:   38.7s
[Parallel(n_jobs=4)]: Done 792 tasks      | elapsed:  1.6min
[Parallel(n_jobs=4)]: Done 1242 tasks      | elapsed:  2.8min
[Parallel(n_jobs=4)]: Done 1792 tasks      | elapsed:  4.4min
[Parallel(n_jobs=4)]: Done 2317 out of 2317 | elapsed:  5.8min finished


2317 variant images | 6.4 min


## 3. Query lists per model

In [5]:
def rows_for(df, prompt, kind, variant="default"):
    out = []
    for r in df.itertuples():
        if variant == "default": a, b = path(r.drawing_A), path(r.drawing_B); pid = r.pair_id
        else: a, b = f"{OUTPUT_DIR}/images/{variant}/{r.drawing_A}.png", f"{OUTPUT_DIR}/images/{variant}/{r.drawing_B}.png"; pid = f"{r.pair_id}__{variant}"
        for order in ("AB", "BA"):
            img1, img2 = (a, b) if order == "AB" else (b, a)
            corr = "same" if r.correct == "same" else ("1" if (r.correct == "A") == (order == "AB") else "2")
            out.append({"pair_id": pid, "order": order, "prompt_id": prompt, "img1": img1, "img2": img2, "correct_img": corr, "kind": kind,
                        "variant": variant, "base_pair": r.pair_id if kind != "mel" else r.base_pair, "cf_delta": getattr(r, "cf_delta", np.nan)})
    return out
PLAN = {}
for m in RENDER_MODELS:
    rows = rows_for(BASE, "P-Def", "render", "default")
    for v in VARIANTS: rows += rows_for(BASE, "P-Def", "render", v)
    for p in RERUNS.get(m, []): rows += rows_for(BASE, p, "rerun")
    PLAN[m] = rows
for m in SIZE_MODELS:
    PLAN[m] = rows_for(BASE, "P-Def", "size") + ([] if m.endswith("int8") else rows_for(MEL, "P-Def", "mel"))
QQ = {}
for m, rows in PLAN.items():
    Q = pd.DataFrame(rows); pr = list(dict.fromkeys(Q.prompt_id))
    Q = Q.sample(frac=1, random_state=SEED); Q["_p"] = Q.prompt_id.map({p: i for i, p in enumerate(pr)}); Q = Q.sort_values("_p", kind="stable").drop(columns="_p")
    Q.to_csv(f"{OUTPUT_DIR}/e4_queries_{m}.csv", index=False); QQ[m] = Q
    print(f"{m}: {len(Q)} queries", Q.groupby(["prompt_id"]).size().to_dict())
if not DRY_RUN: assert all(os.path.exists(p) for Q in QQ.values() for p in set(Q.img1) | set(Q.img2)), "missing images"
if RESUME_ROOT:
    for f in glob.glob(f"{RESUME_ROOT}/**/results/e4/*.jsonl", recursive=True):
        shutil.copy(f, f"{OUTPUT_DIR}/results/e4/{os.path.basename(f)}"); print("resumed", os.path.basename(f))

Q25VL-7B: 3200 queries {'P-Def': 3200}
InternVL-4B: 4400 queries {'P-Def': 3200, 'P-FewShot3': 400, 'P-NoReason': 400, 'P-ReasonFirst': 400}
Q3VL-2B: 3600 queries {'P-Def': 3200, 'P-FewShot3': 400}
InternVL-8B: 1168 queries {'P-Def': 1168}
Q25VL-3B: 1168 queries {'P-Def': 1168}
InternVL-2B: 1168 queries {'P-Def': 1168}
Q25VL-7B-int8: 400 queries {'P-Def': 400}


## 4. Inference worker v0.5

In [6]:
%%writefile /kaggle/working/n10_output/pdcp_infer.py
"""
pdcp_infer.py — PDCP VLM inference worker (v0.5.0). One model per process.
v0.5: more model sizes (Q25VL-3B, InternVL-2B/8B), 8-bit loading, batch cap per prompt (few-shot = 1), longer outputs for P-NoReason/P-ReasonFirst.
v0.4: per-row prompt_id (many prompt variants per process), per-prompt max_new_tokens, few-shot examples (--fewshot).
v0.3: processor_kwargs API, per-model processor options + default batch, 25-word reasons, early abort on unparseable output.

Usage:
  python pdcp_infer.py --model Q3VL-4B --pairs pairs.csv --out results.jsonl --summary sum.json
                       [--prompt P-Def] [--batch 1] [--batch_test 4] [--max_new 160] [--hf_token TOKEN]

pairs.csv columns: pair_id, order (AB|BA), img1, img2 (absolute PNG paths), correct_img ("1"|"2"|"same")
Resumable: rows already in --out (same pair_id+order+prompt) are skipped.
"""
import argparse, gc, json, os, re, sys, time, traceback

MODELS = {   # batch = default batch size on one T4 (from the N04 pilot); proc = extra processor options
    "Q3VL-2B":     {"ids": ["Qwen/Qwen3-VL-2B-Instruct"], "quant": None, "batch": 8, "proc": {}},
    "Q3VL-4B":     {"ids": ["Qwen/Qwen3-VL-4B-Instruct"], "quant": None, "batch": 4, "proc": {}},
    "Q3VL-8B":     {"ids": ["Qwen/Qwen3-VL-8B-Instruct"], "quant": "nf4", "batch": 2, "proc": {}},
    "Q25VL-7B":    {"ids": ["Qwen/Qwen2.5-VL-7B-Instruct"], "quant": "nf4", "batch": 4, "proc": {}},
    "Gemma3-4B":   {"ids": ["google/gemma-3-4b-it"], "quant": None, "batch": 4, "proc": {}},
    "InternVL-4B": {"ids": ["OpenGVLab/InternVL3_5-4B-HF", "OpenGVLab/InternVL3-2B-hf"], "quant": None, "batch": 4,
                    "proc": {"crop_to_patches": False}},   # one 448-px tile per image (pilot: tiling -> 11 s/query, OOM at batch 4)
    "SmolVLM-2B":  {"ids": ["HuggingFaceTB/SmolVLM2-2.2B-Instruct"], "quant": None, "batch": 1, "proc": {}},
    "Q25VL-3B":    {"ids": ["Qwen/Qwen2.5-VL-3B-Instruct"], "quant": None, "batch": 4, "proc": {}},
    "Q25VL-7B-int8": {"ids": ["Qwen/Qwen2.5-VL-7B-Instruct"], "quant": "int8", "batch": 2, "proc": {}},
    "InternVL-2B": {"ids": ["OpenGVLab/InternVL3_5-2B-HF", "OpenGVLab/InternVL3-2B-hf"], "quant": None, "batch": 4, "proc": {"crop_to_patches": False}},
    "InternVL-8B": {"ids": ["OpenGVLab/InternVL3_5-8B-HF", "OpenGVLab/InternVL3-8B-hf"], "quant": "nf4", "batch": 2, "proc": {"crop_to_patches": False}},
}

SYSTEM = ("You are a careful visual analyst. You will be shown two images of node-link diagrams that depict the SAME "
          "network (same nodes and same edges), drawn with different node positions. Answer the question strictly in "
          "the JSON format requested. Do not add any text outside the JSON.")
DEFINITION = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures "
              "how well the straight-line distances between pairs of nodes match the graph-theoretic distances between them "
              "(the number of edges on the shortest path connecting them). A drawing has LOW stress when pairs of nodes that "
              "are few edges apart are drawn close together and pairs that are many edges apart are drawn far apart, "
              "consistently across the whole drawing. A drawing has HIGH stress when these distances disagree.")
JSON_SPEC = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>, '
             '"reason": "<at most 25 words>"}')
QUESTION = "\n\nImage 1 and Image 2 show the same network. Which image has LOWER stress?\n\n"
DEF_NEUTRAL = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures how well "
               "the straight-line distances between ALL pairs of nodes are proportional to their graph-theoretic distances (the number of "
               "edges on the shortest path between them). Only this proportionality matters: the overall size of the drawing, how compact "
               "or spread out it looks, and the absolute length of the edges do not affect stress. A drawing has low stress when the ratio "
               "between drawn distance and path distance is the same for all pairs of nodes.")
STRUCT = ("\n\nEvaluate each image on three criteria, then decide:\n1. Node distribution: are nodes spread evenly, or clumped?\n"
          "2. Edge lengths: are edges of similar length, or do lengths vary a lot?\n3. Edge crossings: how many edges cross each other?\n"
          "Drawings with evenly spread nodes, uniform edge lengths and few crossings usually have lower stress, but judge the overall "
          "correspondence between distances and path lengths.")
J_STRUCT = ('Respond with JSON only:\n{"criteria": {"image1": {"node_distribution": "<short>", "edge_lengths": "<short>", "crossings": "<short>"}, '
            '"image2": {"node_distribution": "<short>", "edge_lengths": "<short>", "crossings": "<short>"}}, '
            '"answer": "1" | "2" | "same", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_NODIFF = ('Respond with JSON only:\n{"answer": "1" | "2" | "no clear difference", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_2AFC = ('You must choose one image. Respond with JSON only:\n{"answer": "1" | "2", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_RFIRST = ('First think step by step in the "reason" field, then give the answer. Respond with JSON only:\n'
            '{"reason": "<at most 60 words>", "answer": "1" | "2" | "same", "confidence": <integer 1-5>}')
J_NOREASON = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>}')
PROMPTS = {
    "P-Def": DEFINITION + QUESTION + JSON_SPEC,
    "P-Expert": ("You are an expert in graph drawing and network visualization. Image 1 and Image 2 show the same network. "
                 "Which image has lower stress?\n\n" + JSON_SPEC),
    "P-NoDef": QUESTION.strip() + "\n\n" + JSON_SPEC,
    "P-DefNeutral": DEF_NEUTRAL + QUESTION + JSON_SPEC,
    "P-Struct": DEFINITION + STRUCT + QUESTION + J_STRUCT,
    "P-NoDiff": DEFINITION + QUESTION + J_NODIFF,
    "P-2AFC": DEFINITION + QUESTION + J_2AFC,
    "P-ReasonFirst": DEFINITION + QUESTION + J_RFIRST,
    "P-NoReason": DEFINITION + QUESTION + J_NOREASON,
    "P-FewShot3": DEFINITION + QUESTION + JSON_SPEC,     # + 3 worked examples inserted before the question
}
MAX_NEW = {"P-Struct": 230, "P-ReasonFirst": 300, "P-NoReason": 96}
BATCH_CAP = {"P-FewShot3": 1}   # 8 images per query: one query at a time (N08 ran out of memory)
FEWSHOT = []   # filled from --fewshot: list of {"img1","img2","answer_json"}


def parse_answer(text):
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = re.search(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', text, re.I)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
        r = re.search(r'"reason"\s*:\s*"([^"]*)', text)
        out["reason"] = r.group(1) if r else None
    return out


def build_messages(img1, img2, prompt_id, with_images=True):
    def im(x):
        return {"type": "image", "image": x} if with_images else {"type": "image"}
    content = [{"type": "text", "text": SYSTEM}]
    if prompt_id == "P-FewShot3" and FEWSHOT:
        content[0]["text"] += "\n\nFirst, three worked examples with the correct answers."
        for k, ex in enumerate(FEWSHOT, 1):
            content += [{"type": "text", "text": f"Example {k}. Image 1:"}, im(ex["img1_pil"]), {"type": "text", "text": "Image 2:"},
                        im(ex["img2_pil"]), {"type": "text", "text": "Correct answer: " + ex["answer_json"]}]
        content.append({"type": "text", "text": "Now the actual question."})
    content += [{"type": "text", "text": "Image 1:"}, im(img1), {"type": "text", "text": "Image 2:"}, im(img2),
                {"type": "text", "text": PROMPTS[prompt_id]}]
    return [{"role": "user", "content": content}]


class Runner:
    def __init__(self, key, hf_token=None, log=print):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
        self.torch, self.key, self.log = torch, key, log
        spec = MODELS[key]
        self.route, self.model, self.proc, self.model_id, self.dtype_name = None, None, None, None, None
        errors = []
        for mid in spec["ids"]:
            for dt in (["float16", "bfloat16"] if spec["quant"] is None else ["float16"]):
                try:
                    t0 = time.time()
                    kw = {"device_map": {"": 0}, "token": hf_token, "low_cpu_mem_usage": True}
                    if spec["quant"] == "int8":
                        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
                    elif spec["quant"] == "nf4":
                        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                                        bnb_4bit_compute_dtype=torch.float16,
                                                                        bnb_4bit_use_double_quant=True)
                    else:
                        kw["dtype"] = getattr(torch, dt)
                    if self.model is None or self.model_id != mid:
                        self._free()
                        self.model = AutoModelForImageTextToText.from_pretrained(mid, **kw).eval()
                        self.proc = AutoProcessor.from_pretrained(mid, token=hf_token)
                    self.model_id, self.dtype_name, self.load_s = mid, (spec["quant"] or dt), time.time() - t0
                    return
                except Exception as e:
                    errors.append(f"{mid} {dt}: {type(e).__name__}: {str(e)[:300]}")
                    self.log("load failed: " + errors[-1])
                    self._free()
        raise RuntimeError("all loads failed | " + " || ".join(errors))

    def reload_dtype(self, dt):
        """Reload the same model in another dtype (used when fp16 produces garbage)."""
        from transformers import AutoModelForImageTextToText
        self._free(keep_proc=True)
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_id, device_map={"": 0},
                                                                 dtype=getattr(self.torch, dt), low_cpu_mem_usage=True).eval()
        self.dtype_name = dt

    def _free(self, keep_proc=False):
        if self.model is not None:
            del self.model
            self.model = None
        if not keep_proc:
            self.proc = None
        gc.collect()
        try:
            self.torch.cuda.empty_cache()
        except Exception:
            pass

    def _inputs(self, batch, prompt_id):
        """batch = list of (img1, img2). Tries route A (chat template loads images) then B (processor call)."""
        torch = self.torch
        routes = [self.route] if self.route else ["A", "B_nested", "B_flat"]
        last = None
        for route in routes:
            try:
                if route == "A":
                    msgs = [build_messages(a, b, prompt_id) for a, b in batch]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    pk = dict(MODELS[self.key].get("proc", {}))
                    if len(batch) > 1:
                        pk["padding"] = True
                    inp = self.proc.apply_chat_template(msgs if len(batch) > 1 else msgs[0], add_generation_prompt=True,
                                                        tokenize=True, return_dict=True, return_tensors="pt",
                                                        processor_kwargs=pk)
                else:
                    texts = [self.proc.apply_chat_template(build_messages(a, b, prompt_id, with_images=False),
                                                           add_generation_prompt=True, tokenize=False) for a, b in batch]
                    ex = [x for e in FEWSHOT for x in (e["img1_pil"], e["img2_pil"])] if prompt_id == "P-FewShot3" else []
                    imgs = [ex + [a, b] for a, b in batch] if route == "B_nested" else [x for ab in batch for x in ex + list(ab)]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    inp = self.proc(text=texts, images=imgs, return_tensors="pt", padding=True, **MODELS[self.key].get("proc", {}))
                self.route = route
                dev = self.model.device
                mdt = torch.float16 if self.dtype_name in ("nf4", "int8", "float16") else getattr(torch, self.dtype_name)
                return {k: (v.to(dev, dtype=mdt) if torch.is_floating_point(v) else v.to(dev)) if hasattr(v, "to") else v
                        for k, v in inp.items()}
            except Exception as e:
                last = e
                if self.route:
                    raise
        raise RuntimeError(f"no input route worked: {type(last).__name__}: {last}")

    def generate(self, batch, prompt_id, max_new=160):
        torch = self.torch
        inp = self._inputs(batch, prompt_id)
        n_in = inp["input_ids"].shape[1]
        with torch.inference_mode():
            out = self.model.generate(**inp, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None)
        texts = self.proc.batch_decode(out[:, n_in:], skip_special_tokens=True)
        return texts, int(n_in)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True); ap.add_argument("--pairs", required=True)
    ap.add_argument("--out", required=True); ap.add_argument("--summary", required=True)
    ap.add_argument("--prompt", default="P-Def"); ap.add_argument("--batch", type=int, default=0, help="0 = model default")
    ap.add_argument("--min_parse", type=float, default=0.5, help="abort if parse rate after 40 queries is below this")
    ap.add_argument("--deadline", type=float, default=0, help="unix time after which the worker stops cleanly")
    ap.add_argument("--batch_test", type=int, default=0); ap.add_argument("--max_new", type=int, default=160)
    ap.add_argument("--hf_token", default=None); ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--fewshot", default=None, help="json list of {img1,img2,answer_json}")
    a = ap.parse_args()
    import pandas as pd, torch
    from PIL import Image
    global FEWSHOT
    def log(s): print(f"[{a.model} {time.strftime('%H:%M:%S')}] {s}", flush=True)
    summ = {"model": a.model, "prompt": a.prompt, "ok": False, "error": None}
    t_start = time.time()
    try:
        P = pd.read_csv(a.pairs)
        if a.limit: P = P.head(a.limit)
        done = set()
        if os.path.exists(a.out):
            for line in open(a.out):
                try:
                    r = json.loads(line); done.add((r["pair_id"], r["order"], r["prompt_id"]))
                except Exception:
                    pass
        if "prompt_id" not in P.columns: P["prompt_id"] = a.prompt
        todo = [r for r in P.to_dict("records") if (r["pair_id"], r["order"], r["prompt_id"]) not in done]
        order_p = {p: i for i, p in enumerate(dict.fromkeys(P.prompt_id))}
        todo.sort(key=lambda r: order_p[r["prompt_id"]])            # stable: keeps the shuffle inside each prompt
        if a.fewshot:
            for e in json.load(open(a.fewshot)):
                e["img1_pil"] = Image.open(e["img1"]).convert("RGB"); e["img2_pil"] = Image.open(e["img2"]).convert("RGB"); FEWSHOT.append(e)
            log(f"loaded {len(FEWSHOT)} few-shot examples")
        log(f"{len(P)} pairs, {len(todo)} to do")
        torch.cuda.reset_peak_memory_stats()
        R = Runner(a.model, a.hf_token, log)
        log(f"loaded {R.model_id} as {R.dtype_name} in {R.load_s:.0f}s")
        summ.update({"model_id": R.model_id, "dtype": R.dtype_name, "load_s": round(R.load_s, 1)})
        load = lambda p: Image.open(p).convert("RGB")

        # warm-up + dtype sanity: if fp16 gives zero parseable answers on 3 items, retry in bf16
        if todo:
            w = todo[:3]
            texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in w[:1]], w[0]["prompt_id"], MAX_NEW.get(w[0]["prompt_id"], a.max_new))
            ok = parse_answer(texts[0])["parse_ok"]
            if not ok:
                t2 = [R.generate([(load(r["img1"]), load(r["img2"]))], r["prompt_id"], MAX_NEW.get(r["prompt_id"], a.max_new))[0][0] for r in w[1:]]
                if not any(parse_answer(t)["parse_ok"] for t in t2) and R.dtype_name == "float16":
                    log("fp16 output unparseable on warm-up -> retrying in bfloat16. sample: " + repr(texts[0][:200]))
                    try:
                        R.reload_dtype("bfloat16"); summ["dtype"] = "bfloat16"
                    except Exception as e:
                        log(f"bf16 reload failed ({type(e).__name__}); reloading fp16 and continuing")
                        R.reload_dtype("float16"); summ["dtype_note"] = "fp16 unparseable, bf16 unavailable"
        summ["route"] = R.route

        f = open(a.out, "a")
        times, n_ok, n_corr, n = [], 0, 0, 0
        bs = a.batch if a.batch > 0 else MODELS[a.model].get("batch", 1)
        summ["batch"] = bs
        groups, cur = [], []
        for r in todo:                       # chunks never mix prompt variants
            if cur and (r["prompt_id"] != cur[0]["prompt_id"] or len(cur) == min(bs, BATCH_CAP.get(cur[0]["prompt_id"], bs))):
                groups.append(cur); cur = []
            cur.append(r)
        if cur: groups.append(cur)
        for chunk in groups:
            pid = chunk[0]["prompt_id"]; mn = MAX_NEW.get(pid, a.max_new)
            if a.deadline and time.time() > a.deadline:
                log("deadline reached - stopping cleanly (resumable)"); summ["stopped_at_deadline"] = True; break
            t0 = time.time()
            def one_by_one(rows):
                out, nin = [], 0
                for r in rows:
                    try:
                        t, nin = R.generate([(load(r["img1"]), load(r["img2"]))], pid, mn); out += t
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache(); out.append(""); log(f"OOM on single query {r['pair_id']} {r['order']} - skipped (recorded as unparsed)")
                return out, nin
            if bs == 1 or len(chunk) == 1:
                texts, n_in = one_by_one(chunk)
            else:
                try:
                    texts, n_in = R.generate([(load(r["img1"]), load(r["img2"])) for r in chunk], pid, mn)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); log("OOM at batch %d -> batch 1 from now on" % bs); bs = 1
                    texts, n_in = one_by_one(chunk)
            dt = (time.time() - t0) / len(chunk)
            for r, t in zip(chunk, texts):
                p = parse_answer(t)
                corr = p["answer"] == str(r["correct_img"]) if p["parse_ok"] else None
                rec = {"pair_id": r["pair_id"], "order": r["order"], "model": a.model, "model_id": R.model_id,
                       "dtype": R.dtype_name, "prompt_id": pid, "raw_text": t, "parsed": p, "parse_ok": p["parse_ok"],
                       "correct_img": str(r["correct_img"]), "correct": corr, "latency_s": round(dt, 3), "input_tokens": n_in,
                       "batch": len(chunk), "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
                f.write(json.dumps(rec) + "\n")
                n += 1; n_ok += p["parse_ok"]; n_corr += bool(corr)
            f.flush(); times.append(dt)
            if n >= 40 and n - len(chunk) < 40 and n_ok / n < a.min_parse:
                log(f"parse rate {n_ok/n:.2f} < {a.min_parse} after {n} queries - aborting this model")
                summ["aborted_low_parse"] = True; break
            if n % 50 < len(chunk):
                log(f"{n}/{len(todo)} | {dt:.2f}s/q | parse {n_ok/n:.2f} | acc {n_corr/max(1,n_ok):.2f}")
        f.close()
        summ.update({"n": n, "sec_per_query": round(float(sum(times) / max(1, len(times))), 3), "parse_rate": round(n_ok / max(1, n), 3),
                     "acc_on_parsed": round(n_corr / max(1, n_ok), 3), "input_tokens": n_in if todo else None})

        # optional batched-throughput test on the first items (not written to results)
        if a.batch_test and a.batch_test > 1 and len(P) >= a.batch_test * 3:
            rows = P.to_dict("records")[: a.batch_test * 3]
            try:
                t0 = time.time()
                for i in range(0, len(rows), a.batch_test):
                    texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in rows[i:i + a.batch_test]], a.prompt, a.max_new)
                summ["batch_test"] = {"batch": a.batch_test, "sec_per_query": round((time.time() - t0) / len(rows), 3),
                                      "parse_rate": round(sum(parse_answer(t)["parse_ok"] for t in texts) / len(texts), 3)}
            except Exception as e:
                summ["batch_test"] = {"batch": a.batch_test, "error": f"{type(e).__name__}: {str(e)[:200]}"}
            log(f"batch test: {summ['batch_test']}")
        summ["vram_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
        summ["ok"] = True
    except Exception as e:
        summ["error"] = f"{type(e).__name__}: {str(e)[:1500]}"
        log("FAILED: " + summ["error"]); traceback.print_exc()
    summ["wall_s"] = round(time.time() - t_start, 1)
    json.dump(summ, open(a.summary, "w"), indent=2)
    log(f"done: {json.dumps({k: summ.get(k) for k in ['ok', 'n', 'sec_per_query', 'parse_rate', 'acc_on_parsed', 'vram_peak_gb']})}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/n10_output/pdcp_infer.py


## 5. Run (two GPUs; longest model first)

In [7]:
HF_CACHE = "/tmp/hf_cache"
sys.path.insert(0, OUTPUT_DIR); import pdcp_infer as I; importlib.reload(I)
DEADLINE = T0 + SESSION_HOURS * 3600
ORDER = ["Q25VL-7B", "InternVL-4B", "Q3VL-2B", "InternVL-8B", "Q25VL-3B", "InternVL-2B", "Q25VL-7B-int8"]
def ndone(m):
    f = f"{OUTPUT_DIR}/results/e4/{m}.jsonl"; return sum(1 for _ in open(f)) if os.path.exists(f) else 0
if DRY_RUN:
    rng = np.random.default_rng(0)
    for m, Q in QQ.items():
        with open(f"{OUTPUT_DIR}/results/e4/{m}.jsonl", "w") as f:
            for r in Q.to_dict("records"):
                a = rng.choice(["1", "2", "same"], p=[0.5, 0.45, 0.05]) if rng.random() > 0.03 else None
                p = {"answer": a, "confidence": 4, "reason": "x", "parse_ok": a is not None}
                f.write(json.dumps({"pair_id": r["pair_id"], "order": r["order"], "model": m, "prompt_id": r["prompt_id"], "raw_text": "", "parsed": p,
                                    "parse_ok": a is not None, "correct_img": r["correct_img"]}) + "\n")
queue = [m for m in ORDER if m in QQ and ndone(m) < len(QQ[m]) and not DRY_RUN]; running = {}
def launch(model, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), HF_HUB_CACHE=HF_CACHE, TOKENIZERS_PARALLELISM="false", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    cmd = [sys.executable, f"{OUTPUT_DIR}/pdcp_infer.py", "--model", model, "--pairs", f"{OUTPUT_DIR}/e4_queries_{model}.csv",
           "--out", f"{OUTPUT_DIR}/results/e4/{model}.jsonl", "--summary", f"{OUTPUT_DIR}/results/e4/{model}.summary.json",
           "--max_new", "96", "--deadline", str(DEADLINE), "--fewshot", f"{OUTPUT_DIR}/fewshot.json", "--min_parse", "0.3"]
    logf = open(f"{OUTPUT_DIR}/logs/{model}.log", "a")
    running[gpu] = (model, subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env), logf)
    print(f"[{(time.time()-T0)/60:6.1f} min] start {model} on GPU {gpu}", flush=True)
last = 0
while queue or running:
    for gpu in range(N_GPU):
        if gpu not in running and queue and time.time() < DEADLINE - 1800: launch(queue.pop(0), gpu)
    if not running: break
    time.sleep(30)
    if time.time() - last > 1800:
        last = time.time(); print("   " + " | ".join(f"{m}: {ndone(m)}/{len(QQ[m])}" for m in QQ), flush=True)
    for gpu, (model, p, logf) in list(running.items()):
        if p.poll() is not None:
            logf.close(); del running[gpu]
            print(f"[{(time.time()-T0)/60:6.1f} min] end {model}: {open(f'{OUTPUT_DIR}/logs/{model}.log').read().strip().splitlines()[-1][-250:]}", flush=True)
            for mid in I.MODELS[model]["ids"]: shutil.rmtree(f"{HF_CACHE}/models--{mid.replace('/', '--')}", ignore_errors=True)
status = {m: {"done": ndone(m), "total": len(QQ[m])} for m in QQ}; status["all_complete"] = all(v["done"] >= v["total"] for v in status.values())
json.dump(status, open(f"{OUTPUT_DIR}/e4_status.json", "w"), indent=2); print(status)
print(f"run phase finished after {(time.time()-T0)/3600:.2f} h")

[   6.5 min] start Q25VL-7B on GPU 0
[   6.5 min] start InternVL-4B on GPU 1
   Q25VL-7B: 0/3200 | InternVL-4B: 0/4400 | Q3VL-2B: 0/3600 | InternVL-8B: 0/1168 | Q25VL-3B: 0/1168 | InternVL-2B: 0/1168 | Q25VL-7B-int8: 0/400
   Q25VL-7B: 228/3200 | InternVL-4B: 904/4400 | Q3VL-2B: 0/3600 | InternVL-8B: 0/1168 | Q25VL-3B: 0/1168 | InternVL-2B: 0/1168 | Q25VL-7B-int8: 0/400
   Q25VL-7B: 480/3200 | InternVL-4B: 1900/4400 | Q3VL-2B: 0/3600 | InternVL-8B: 0/1168 | Q25VL-3B: 0/1168 | InternVL-2B: 0/1168 | Q25VL-7B-int8: 0/400
   Q25VL-7B: 748/3200 | InternVL-4B: 2900/4400 | Q3VL-2B: 0/3600 | InternVL-8B: 0/1168 | Q25VL-3B: 0/1168 | InternVL-2B: 0/1168 | Q25VL-7B-int8: 0/400
   Q25VL-7B: 1012/3200 | InternVL-4B: 3368/4400 | Q3VL-2B: 0/3600 | InternVL-8B: 0/1168 | Q25VL-3B: 0/1168 | InternVL-2B: 0/1168 | Q25VL-7B-int8: 0/400
   Q25VL-7B: 1268/3200 | InternVL-4B: 3620/4400 | Q3VL-2B: 0/3600 | InternVL-8B: 0/1168 | Q25VL-3B: 0/1168 | InternVL-2B: 0/1168 | Q25VL-7B-int8: 0/400
   Q25VL-7B: 1528/320

## 6. Analysis

In [8]:
def load(root, sub, prompt=None):
    R = []
    for f in glob.glob(f"{root}/**/results/{sub}/*.jsonl", recursive=True): R += [json.loads(l) for l in open(f)]
    R = pd.DataFrame(R)
    if not len(R): return R
    if "prompt_id" not in R or prompt: R["prompt_id"] = R["prompt_id"] if "prompt_id" in R else prompt
    a = R.parsed.map(lambda p: (p or {}).get("answer")); R["ans"] = a
    R["choice"] = np.where(a == "same", "same", np.where(a.isin(["1", "2"]), np.where((a == "1") == (R.order == "AB"), "A", "B"), None))
    R["pa"] = R.choice.map({"A": 1.0, "B": 0.0, "same": 0.5})
    return R
E4 = load(OUTPUT_DIR, "e4"); E1 = load(N05_ROOT, "e1", "P-Def"); E2 = load(N06_ROOT, "e2", "P-Def")
key = dict(zip(PR.pair_id, PR.correct)); key.update(dict(zip(MCP.pair_id, MCP.correct)))
E4["base_pair"] = E4.pair_id.str.split("__").str[0]; E4["variant"] = np.where(E4.pair_id.str.contains("__"), E4.pair_id.str.split("__").str[1], "default")
for R in [E1, E2]:
    if len(R): R["base_pair"] = R.pair_id; R["variant"] = "default"
Bi = BASE.set_index("pair_id")
dmel = {p: MELV[Bi.loc[p, "drawing_B"]] - MELV[Bi.loc[p, "drawing_A"]] for p in Bi.index}
def pair_pa(R):   # order-averaged P(choose A) per (model, prompt, pair)
    return R[R.choice.notna()].groupby(["model", "prompt_id", "pair_id"]).pa.mean()
def stats(R, ids):
    d = R[R.pair_id.isin(ids)]; ok = d[d.choice.notna()]; b = ok.base_pair.map(Bi.dstress_design); c = ok.base_pair.map(Bi.cond)
    k = ok.base_pair.map(key); nz = ok[(b > 0).values]
    pa = ok.groupby("pair_id").pa.mean(); pbs = pd.Series([p.split("__")[0] for p in pa.index], index=pa.index)
    z = pa[pbs.map(Bi.dstress_design) == 0]; zc = pbs.map(Bi.cond)[z.index]
    return {"n": len(d), "parse": round(float(d.choice.notna().mean()), 3), "acc_dstress>0": round(float((nz.choice == nz.base_pair.map(key)).mean()), 3),
            "p_image1": round(float((ok.ans == "1").mean()), 3), "p_same": round(float((ok.ans == "same").mean()), 3),
            "lure_gap": round(float(z[zc == "congruent"].mean() - z[zc == "conflict_all"].mean()), 3) if len(z) else np.nan}

# ---- E4a rendering ----
rows = []
for m in RENDER_MODELS:
    R = E4[(E4.model == m) & (E4.prompt_id == "P-Def")]
    if not len(R): continue
    pa = pair_pa(R).droplevel([0, 1]); base_pa = pa[pa.index.isin(BASE.pair_id)]
    ref = pair_pa(E1[(E1.model == m) & E1.pair_id.isin(BASE.pair_id)]).droplevel([0, 1]) if len(E1) else pd.Series(dtype=float)
    common = base_pa.index.intersection(ref.index)
    floor = float((base_pa[common] - ref[common]).abs().mean()) if len(common) else np.nan
    floor_flip = float(((base_pa[common] > 0.5) != (ref[common] > 0.5)).mean()) if len(common) else np.nan
    for v in ["default"] + list(VARIANTS):
        ids = list(BASE.pair_id) if v == "default" else [f"{p}__{v}" for p in BASE.pair_id]
        s = stats(R, ids); pv = pa[pa.index.isin(ids)]; pv.index = [i.split("__")[0] for i in pv.index]; cm = pv.index.intersection(base_pa.index)
        rows.append({"model": m, "variant": v, **s, "instability": round(float((pv[cm] - base_pa[cm]).abs().mean()), 3) if v != "default" else 0.0,
                     "decision_change": round(float(((pv[cm] > 0.5) != (base_pa[cm] > 0.5)).mean()), 3) if v != "default" else 0.0,
                     "noise_floor_instability": round(floor, 3), "noise_floor_decision_change": round(floor_flip, 3)})
RS = pd.DataFrame(rows); RS.to_csv(f"{OUTPUT_DIR}/e4_render_summary.csv", index=False)
pd.set_option("display.width", 220); print(RS.to_string(index=False))

# ---- E4b size / quantisation (with E1/E2 models) ----
def size_row(m, Rb, Rm):
    s = stats(Rb, list(BASE.pair_id)); pb = pair_pa(Rb[Rb.pair_id.isin(BASE.pair_id)]).droplevel([0, 1])
    out = {"model": m, "family": FAMILY.get(m, "?"), "params_B": PARAMS_B.get(m, np.nan), "quant": "int8" if m.endswith("int8") else ("nf4" if m in ("Q25VL-7B", "InternVL-8B") else "fp16"), **s}
    if len(Rm):
        pm = pair_pa(Rm[Rm.pair_id.isin(MEL.pair_id)]).droplevel([0, 1]); mm = MEL.set_index("pair_id")
        bv = pd.Series(pm.index, index=pm.index).map(mm.base_pair).map(pb).astype(float)
        sh = (pm.astype(float) - bv).dropna(); dl = pd.Series(sh.index, index=sh.index).map(mm.cf_delta).astype(float).values
        for d in MEL_DELTAS: out[f"mel_shift_{d:+.2f}"] = round(float(sh[dl == d].mean()), 3)
        out["compactness_bias"] = round(out[f"mel_shift_{MEL_DELTAS[0]:+.2f}"] - out[f"mel_shift_{MEL_DELTAS[1]:+.2f}"], 3)
    # cue weights on the 200 base pairs (standardized regression of P(A) on Δstress and Δcues)
    X = pd.DataFrame({"ksm": Bi.d_ksm, "nu": Bi.d_nu, "eld": Bi.d_eld, "xr": Bi.d_xr, "mel": pd.Series(dmel)}).loc[pb.index.intersection(Bi.index)]
    if len(X) > 20:
        Z = (X - X.mean()) / X.std(); y = pb[X.index]; y = (y - y.mean()) / (y.std() + 1e-12)
        bta = np.linalg.lstsq(np.c_[np.ones(len(Z)), Z.values], y.values, rcond=None)[0][1:]; out.update({f"beta_{k}": round(float(v), 3) for k, v in zip(X.columns, bta)})
    return out
srows = []
for m in sorted(set(E1.model) if len(E1) else set()):
    if m in PARAMS_B: srows.append({**size_row(m, E1[E1.model == m], E2[E2.model == m] if len(E2) else pd.DataFrame()), "source": "E1/E2"})
for m in SIZE_MODELS:
    R = E4[E4.model == m]
    if len(R): srows.append({**size_row(m, R, R), "source": "N10"})
SS = pd.DataFrame(srows); SS.to_csv(f"{OUTPUT_DIR}/e4_size_summary.csv", index=False); print(SS.to_string(index=False))

# ---- reruns ----
rr = []
for m, prompts in RERUNS.items():
    for p in ["P-Def"] + prompts:
        R = E4[(E4.model == m) & (E4.prompt_id == p) & E4.pair_id.isin(BASE.pair_id)]
        if len(R): rr.append({"model": m, "prompt": p, **stats(R, list(BASE.pair_id)),
                              "mean_conf": round(float(pd.to_numeric(R.parsed.map(lambda q: (q or {}).get("confidence")), errors="coerce").mean()), 2)})
RR = pd.DataFrame(rr); RR.to_csv(f"{OUTPUT_DIR}/e4_rerun_summary.csv", index=False); print(RR.to_string(index=False))

      model    variant   n  parse  acc_dstress>0  p_image1  p_same  lure_gap  instability  decision_change  noise_floor_instability  noise_floor_decision_change
   Q25VL-7B    default 400    1.0          0.457     0.182   0.000     0.229        0.000            0.000                    0.083                        0.060
   Q25VL-7B     res384 400    1.0          0.348     0.335   0.000     0.604        0.228            0.160                    0.083                        0.060
   Q25VL-7B    res1024 400    1.0          0.519     0.098   0.000     0.146        0.135            0.110                    0.083                        0.060
   Q25VL-7B node_small 400    1.0          0.478     0.195   0.000    -0.021        0.107            0.090                    0.083                        0.060
   Q25VL-7B node_large 400    1.0          0.435     0.122   0.000     0.062        0.100            0.075                    0.083                        0.060
   Q25VL-7B  edge_thin 400    1.0 

## 7. Figures

In [9]:
if len(RS):
    fig, ax = plt.subplots(1, 3, figsize=(19, 4.2))
    for a, col, t in zip(ax, ["acc_dstress>0", "instability", "p_image1"], ["Accuracy (Δstress > 0)", "Instability vs default image (|ΔP(A)|)", "P(answer 'Image 1')"]):
        pv = RS.pivot(index="variant", columns="model", values=col).reindex(["default"] + list(VARIANTS)); pv.plot.bar(ax=a, rot=30); a.set_title(t); a.set_xlabel("")
    for m, c in zip(RS.model.unique(), plt.rcParams["axes.prop_cycle"].by_key()["color"]):
        f = RS[RS.model == m].noise_floor_instability.iloc[0]
        if np.isfinite(f): ax[1].axhline(f, ls="--", c=c, lw=1)
    ax[0].axhline(0.5, ls=":", c="grey"); ax[1].text(0.01, 0.95, "dashed = re-run noise floor (vs E1)", transform=ax[1].transAxes, fontsize=8, va="top")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_e4_render.png", dpi=125); plt.show()
if not DRY_RUN:
    r0 = BASE.iloc[0]; fig, ax = plt.subplots(1, 8, figsize=(24, 3.4))
    for a, v in zip(ax, ["default"] + list(VARIANTS)):
        p = path(r0.drawing_A) if v == "default" else f"{OUTPUT_DIR}/images/{v}/{r0.drawing_A}.png"
        a.imshow(Image.open(p).convert("RGB").resize((384, 384))); a.set_title(v); a.axis("off")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_e4_examples.png", dpi=100); plt.show()
if len(SS):
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.2)); mk = {"Qwen3-VL": "o", "Qwen2.5-VL": "s", "InternVL3.5": "^"}
    for a, col, t in zip(ax, ["acc_dstress>0", "compactness_bias", "p_image1"], ["Accuracy (Δstress > 0)", "Compactness bias (MEL −0.05 minus +0.10)", "P(answer 'Image 1')"]):
        if col not in SS: continue
        for fam, g in SS.groupby("family"):
            g = g.sort_values("params_B"); a.plot(g.params_B, g[col], marker=mk.get(fam, "o"), label=fam)
            for i, r in g.iterrows(): a.annotate(r["quant"], (r["params_B"], r[col]), fontsize=7, xytext=(3, 3), textcoords="offset points")
        a.set_xlabel("parameters (B)"); a.set_title(t); a.legend(fontsize=7)
    ax[0].axhline(0.5, ls=":", c="grey"); ax[1].axhline(0, c="k", lw=.8)
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_e4_size.png", dpi=125); plt.show()
if len(RR):
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
    for a, col in zip(ax, ["acc_dstress>0", "p_image1", "parse"]):
        RR.pivot(index="prompt", columns="model", values=col).plot.bar(ax=a, rot=20); a.set_title(col); a.set_xlabel("")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_e4_reruns.png", dpi=125); plt.show()

## 8. Manifest

In [10]:
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N10 E4 · {(time.time()-T0)/3600:.2f} h · complete: {status['all_complete']} · " + " | ".join(f"{m}: {status[m]['done']}/{status[m]['total']}" for m in QQ), ""]
for root, _, files in os.walk(OUTPUT_DIR):
    if "/images" in root or "fewshot_img" in root: continue
    for f in sorted(files):
        p = os.path.join(root, f); lines.append(f"{os.path.relpath(p, OUTPUT_DIR):40s} {os.path.getsize(p)/1024:9.1f} KB")
lines.append(f"images/: {sum(len(os.listdir(f'{OUTPUT_DIR}/images/{v}')) for v in VARIANTS)} PNG")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))
print("ALL COMPLETE" if status["all_complete"] else "NOT COMPLETE -> save output as dataset, set RESUME_ROOT, run again")

N10 E4 · 7.42 h · complete: False · Q25VL-7B: 3200/3200 | InternVL-4B: 4400/4400 | Q3VL-2B: 3600/3600 | InternVL-8B: 1168/1168 | Q25VL-3B: 1168/1168 | InternVL-2B: 1168/1168 | Q25VL-7B-int8: 0/400

e4_queries_InternVL-2B.csv                   278.1 KB
e4_queries_InternVL-4B.csv                   839.8 KB
e4_queries_InternVL-8B.csv                   278.1 KB
e4_queries_Q25VL-3B.csv                      278.1 KB
e4_queries_Q25VL-7B-int8.csv                  90.5 KB
e4_queries_Q25VL-7B.csv                      560.4 KB
e4_queries_Q3VL-2B.csv                       653.2 KB
e4_render_summary.csv                          1.8 KB
e4_rerun_summary.csv                           0.4 KB
e4_size_summary.csv                            0.9 KB
e4_status.json                                 0.4 KB
fewshot.json                                   0.9 KB
fig_e4_examples.png                          295.1 KB
fig_e4_render.png                             75.0 KB
fig_e4_reruns.png                             